# Quantum (VQC) model: minimal training run

The default model of `configs/vqc.yaml`: post-selected circuits (2 ansatz layers per word, 1 qubit per noun wire,
3 output qubits) against a 2-layer image feature map, trained with the Fubini-Study InfoNCE loss on SVO-Probes.
Needs the processed data (`python scripts/retrieve.py -cfg configs/vqc.yaml`). For full runs use `scripts/train.py`.

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')   # paths in the configs are relative to the repository root

from modules.config import setup_exp
from modules.factory import load_experiment
from modules.models.fusion.trainer import ContrastiveTrainer
from modules.models.fusion.evaluator import MMEvaluator
from modules.factory import build_optimizer

config, device, seed = setup_exp('configs/vqc.yaml', seed=0, overrides=['epochs=3', 'warmup_epochs=1'])
exp = load_experiment(config, device)
print(f"{len(exp.train)} training and {len(exp.val)} validation captions on {device}")

In [ ]:
optimizer, scheduler = build_optimizer(config, exp.text_model, exp.image_model)
trainer = ContrastiveTrainer(exp.image_model, exp.text_model, optimizer, exp.loss_fn, device)
evaluator = MMEvaluator(exp.image_model, exp.text_model, device)
train_loader = exp.data.get_loader(exp.train, split='train')
val_loader = exp.data.get_loader(exp.val, split='val')

for epoch in range(config['epochs']):
    loss = trainer.train_epoch(train_loader, exp.data.eval_mapper)
    scheduler.step()
    metrics = evaluator.eval_set(val_loader, config['dataset']['val']['diagnostics'], exp.data.eval_mapper)
    print(f"epoch {epoch} | loss {loss:.4f} | val hard_neg_acc {metrics['hard_neg_acc']:.4f}")

In [ ]:
# test sets: std (choose the image, 1833 pairs), swap (95) and swap_full (1777): choose the caption
results, _ = evaluator.benchmark(exp.data, config['dataset']['test'])
{k: round(v, 4) for k, v in results.items() if k.endswith('hard_neg_acc')}